# NIST TN 1822 — Verif.5.2: Maximum flow rates (Mode B - emergent flow)

100 agents in an 8 m x 5 m room with a 1 m exit. NIST Mode B: let flow emerge from agent dynamics and verify it stays below the 1.33 p/m/s threshold (IMO MSC/Circ.1238). To switch to Mode A (throttled exit), set `enable_throughput_throttling=True` on the exit.

In [ ]:
from datetime import datetime
print(f"Executed on {datetime.now().astimezone().strftime('%d %B %Y, %H:%M %Z')}")

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import pedpy
from shapely.geometry import Point, Polygon

from jupedsim_scenarios import load_scenario, run_scenario

In [ ]:
plt.rcParams.update({
    "figure.facecolor": "white",
    "axes.facecolor": "#f7f7f5",
    "axes.edgecolor": "#3a3a3a",
    "axes.labelcolor": "#1d1d1d",
    "axes.titleweight": "bold",
    "axes.grid": True,
    "grid.alpha": 0.3,
    "font.size": 11,
    "figure.figsize": (8, 5),
})

## Load and run

In [ ]:
SCENARIO_ZIP = Path("scenario_files") / "Nist-5-2-max-flow.zip"
scenario = load_scenario(str(SCENARIO_ZIP))
print(scenario.summary())
print('exit fields:', list(scenario.exits['jps-exits_0'].keys()))
result = run_scenario(scenario, seed=42)
df = result.trajectory_dataframe()

## Compute exit flow rate

Count agents crossing the x = 0 line per second of wall-clock time.

In [ ]:
EXIT_WIDTH_M = 1.0
THRESHOLD_PMS = 1.33
rows = []
for agent_id, sub in df.sort_values(['id', 'frame']).groupby('id'):
    crossed = sub[sub.x <= 0.3]
    if len(crossed):
        rows.append({'agent': int(agent_id), 'exit_time_s': crossed.iloc[0].frame / result.frame_rate})
exits_df = pd.DataFrame(rows).sort_values('exit_time_s').reset_index(drop=True)
exits_df['cum'] = np.arange(1, len(exits_df) + 1)

window = 1.0
flow_rows = []
for t_end in np.arange(window, exits_df.exit_time_s.max() + window, window):
    count = ((exits_df.exit_time_s > t_end - window) & (exits_df.exit_time_s <= t_end)).sum()
    flow_rows.append({'t_end_s': t_end, 'flow_p_per_m_s': count / EXIT_WIDTH_M / window})
flow = pd.DataFrame(flow_rows)

## Plot flow vs time

In [ ]:
fig, ax = plt.subplots()
ax.plot(flow.t_end_s, flow.flow_p_per_m_s, label='measured')
ax.axhline(THRESHOLD_PMS, color='r', ls='--', label=f'threshold {THRESHOLD_PMS} p/m/s')
ax.set_xlabel('time [s]'); ax.set_ylabel('flow [p/m/s]')
ax.legend()
plt.show()

## Acceptance (Mode B)

In [ ]:
peak_flow = flow.flow_p_per_m_s.max()
print(f'peak emergent flow = {peak_flow:.3f} p/m/s; threshold = {THRESHOLD_PMS} p/m/s')
# Mode B is validation, not a hard pass/fail: we report only.
if peak_flow > THRESHOLD_PMS:
    print('WARNING: emergent flow exceeds the IMO reference threshold')
result.cleanup()